# Praktik Week 6: Custom YOLOv8 Training, Evaluation & Video Interface

Selamat datang di **Week 6**! Pada minggu ini, kita akan menyelesaikan alur kerja (*end-to-end workflow*) pelatihan **YOLOv8** dengan dataset kustom dari **Roboflow**, melakukan evaluasi performa model secara mendalam, serta membangun **Antarmuka Pemrosesan Video** (Video Interface) berbasis OpenCV dan Gradio.

### 🎯 Tujuan Pembelajaran:
1. Mengunduh dataset kustom `reels-moi4j` (versi 4) dari Roboflow API.
2. Memverifikasi struktur dataset dan berkas `data.yaml`.
3. Melakukan *Transfer Learning / Fine-tuning* model **YOLOv8 Nano** (`yolov8n.pt`).
4. Menganalisis metrik evaluasi model (Precision, Recall, mAP@0.5, mAP@0.5:0.95, Confusion Matrix, dan Loss Curves).
5. Menguji inferensi deteksi objek pada sampel gambar uji (*test set / validation set*).
6. Membangun pipeline pemrosesan video asli dengan anotasi *bounding box*, label kelas, nilai *confidence*, dan indikator FPS *real-time*.
7. Mengembangkan **Antarmuka Interaktif (Web Interface)** berbasis **Gradio** untuk pengujian video secara dinamis.

## Langkah 1: Instalasi Library Pendukung

Kita mengunduh library yang dibutuhkan: `ultralytics` untuk YOLOv8, `roboflow` untuk mengunduh dataset, `opencv-python` untuk pemrosesan video, `matplotlib` & `seaborn` untuk visualisasi, serta `gradio` untuk antarmuka web interaktif.

In [ ]:
# Instalasi dependensi utama
!pip install ultralytics roboflow opencv-python matplotlib pandas seaborn gradio

## Langkah 2: Import Library & Pemeriksaan Environment Hardware

Memastikan GPU (CUDA) terdeteksi agar proses pelatihan berjalan cepat. Jika GPU tidak tersedia, sistem akan otomatis menggunakan CPU.

In [ ]:
import torch
import ultralytics
from ultralytics import YOLO
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import os
import yaml
import time

print(f"PyTorch Version: {torch.__version__}")
print(f"Ultralytics Version: {ultralytics.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Warning: Running on CPU. Pelatihan model mungkin membutuhkan waktu lebih lama.")
    print("💡 Tip: Pada CPU, Anda dapat mengatur `epochs=3` atau `imgsz=320` untuk pengujian cepat.")

## Langkah 3: Unduh Dataset Kustom dari Roboflow

Kita menggunakan Roboflow API untuk mengunduh dataset `reels-moi4j` versi 4 dari workspace `visionamarine` dengan format **YOLOv8**.

In [ ]:
from roboflow import Roboflow

# Inisialisasi Roboflow API
rf = Roboflow(api_key="q294SmqNM6eLolt7AeUe")
project = rf.workspace("visionamarine").project("reels-moi4j")
version = project.version(4)

# Download dataset format yolov8
dataset = version.download("yolov8")
print(f"Dataset berhasil diunduh di lokasi: {dataset.location}")

## Langkah 4: Verifikasi & Penyesuaian `data.yaml`

Berkas `data.yaml` memuat jalur direktori gambar (*train*, *val*, *test*), jumlah kelas (`nc`), dan nama kelas (`names`). Kita pastikan jalur absolut/relatif dikonfigurasi dengan benar untuk Ultralytics YOLOv8.

In [ ]:
# Tentukan path data.yaml
yaml_path = os.path.join(dataset.location, "data.yaml")

# Baca isi data.yaml
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

# Perbarui path direktori agar menggunakan path absolut
data_config['path'] = os.path.abspath(dataset.location)

with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f)

print("--- Konfigurasi Dataset (data.yaml) ---")
print(f"Path Utama: {data_config.get('path')}")
print(f"Jumlah Kelas (nc): {data_config.get('nc')}")
print(f"Nama Kelas (names): {data_config.get('names')}")

## Langkah 5: Pelatihan Model (Custom Training YOLOv8)

Kita memulai proses *Transfer Learning* dengan memilih bobot awal **YOLOv8 Nano** (`yolov8n.pt`).

### Hyperparameter Utama:
- `data`: Path ke file `data.yaml`
- `epochs`: Jumlah iterasi (misal: 25 untuk training penuh, atau 3-5 untuk tes cepat di CPU)
- `imgsz`: 640x640 piksel (atau 320x320 untuk CPU)
- `batch`: 16 (atau 32)
- `name`: Nama folder eksperimen hasil latihan (`reels_v4_experiment`)

In [ ]:
# Load Pre-trained Model YOLOv8 Nano
model = YOLO('yolov8n.pt')

# Pilih jumlah epoch berdasarkan device (GPU = 25, CPU = 3 untuk demonstrasi cepat)
epochs_setting = 25 if torch.cuda.is_available() else 3
imgsz_setting = 640 if torch.cuda.is_available() else 320

print(f"Memulai training dengan {epochs_setting} epochs, imgsz={imgsz_setting}...")

# Jalankan Pelatihan (Training)
results = model.train(
    data=yaml_path,
    epochs=epochs_setting,
    imgsz=imgsz_setting,
    batch=16,
    name='reels_v4_experiment',
    exist_ok=True,
    plots=True
)

print("Proses pelatihan selesai! Hasil disimpan di folder runs/detect/reels_v4_experiment")

## Langkah 6: Evaluasi Kuantitatif Metrik Performa Model

Setelah pelatihan selesai, kita memuat bobot terbaik (`best.pt`) dan menjalankan validasi pada *validation set* untuk mendapatkan nilai **Precision**, **Recall**, **mAP@0.5**, dan **mAP@0.5:0.95**.

In [ ]:
# Pencarian dinamis berkas bobot terbaik
candidate_paths = [
    os.path.join('runs', 'detect', 'reels_v4_experiment', 'weights', 'best.pt'),
    os.path.join('runs', 'detect', 'train', 'weights', 'best.pt'),
    os.path.join('best.pt'),
    os.path.join('..', 'week5', 'best.pt'),
    'yolov8n.pt'
]

best_model_path = next((p for p in candidate_paths if os.path.exists(p)), 'yolov8n.pt')
print(f"✅ Menggunakan bobot model: {best_model_path}")

# Load model
best_model = YOLO(best_model_path)

# Jalankan Evaluasi Validasi
metrics = best_model.val(data=yaml_path)

print("\n================ HASIL EVALUASI MODEL ================")
print(f"Precision (P)   : {metrics.box.mp:.4f}")
print(f"Recall (R)      : {metrics.box.mr:.4f}")
print(f"mAP@0.5         : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95    : {metrics.box.map:.4f}")
print("=======================================================")

## Langkah 7: Visualisasi Grafis (Loss Curves & Confusion Matrix)

Menampilkan grafik hasil pelatihan dan evaluasi yang berada di direktori `runs/detect/`.

In [ ]:
# Pencarian otomatis seluruh grafik hasil training di runs/
target_filenames = ['results.png', 'confusion_matrix.png', 'F1_curve.png', 'PR_curve.png', 'BoxF1_curve.png', 'BoxPR_curve.png', 'labels.jpg']

found_plots = {}
for root, dirs, files in os.walk('runs'):
    for file in files:
        if file in target_filenames and file not in found_plots:
            found_plots[file] = os.path.join(root, file)

# Tampilkan grafik yang ditemukan
if found_plots:
    plot_items = list(found_plots.items())[:4]
    plt.figure(figsize=(16, 12))
    for i, (name, path) in enumerate(plot_items):
        img = cv.imread(path)
        if img is not None:
            img_rgb = cv.cvtColor(img, cv.COLOR_BGR2RGB)
            plt.subplot(2, 2, i+1)
            plt.imshow(img_rgb)
            plt.title(f"{name} ({os.path.basename(os.path.dirname(path))})", fontsize=12, fontweight='bold')
            plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("Info: Grafik pelatihan belum ditemukan di direktori runs/. Selesaikan training untuk menghasilkan grafik.")

## Langkah 8: Pengujian Inferensi pada Sampel Gambar Test/Validation Set

Mari kita jalankan deteksi pada beberapa gambar sampel dan menampilkan hasilnya.

In [ ]:
# Cari folder gambar pengujian (test/images, valid/images, atau train/images)
candidate_img_dirs = [
    os.path.join(dataset.location, 'test', 'images'),
    os.path.join(dataset.location, 'valid', 'images'),
    os.path.join(dataset.location, 'train', 'images')
]

test_images_dir = next((d for d in candidate_img_dirs if os.path.exists(d)), None)

if test_images_dir:
    sample_images = [os.path.join(test_images_dir, f) for f in os.listdir(test_images_dir) if f.endswith(('.jpg', '.png', '.jpeg'))][:4]
    if sample_images:
        plt.figure(figsize=(15, 10))
        for i, img_path in enumerate(sample_images):
            results = best_model(img_path, conf=0.25)
            res_plotted = results[0].plot()
            res_rgb = cv.cvtColor(res_plotted, cv.COLOR_BGR2RGB)
            
            plt.subplot(2, 2, i+1)
            plt.imshow(res_rgb)
            plt.title(f"Sample {i+1} ({os.path.basename(img_path)[:20]}...)", fontsize=10)
            plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"Folder '{test_images_dir}' tidak berisi gambar.")
else:
    print("Folder gambar tidak ditemukan.")

## Langkah 9: Pipeline Pemrosesan Video Asli (OpenCV + YOLOv8)

Di langkah ini, kita membuat fungsi pemrosesan video yang:
1. Membaca berkas video input (`.mp4`, `.avi`, dll).
2. Memproses frame demi frame menggunakan model hasil pelatihan (`best_model_path`).
3. Menggambar *bounding box*, label kelas, dan tingkat kepercayaan (*confidence*).
4. Menghitung dan menampilkan **FPS (Frames Per Second)** secara *real-time* pada video.
5. Menyimpan video hasil anotasi ke berkas keluaran.

In [ ]:
def process_video_pipeline(input_video_path, output_video_path, model_path=best_model_path, conf_thresh=0.3):
    """
    Memproses video frame-by-frame dengan YOLOv8 dan memberikan anotasi visual real-time.
    """
    if not os.path.exists(input_video_path):
        print(f"Error: File video '{input_video_path}' tidak ditemukan!")
        return None

    # Load Model
    model = YOLO(model_path)
    
    # Buka Video Stream
    cap = cv.VideoCapture(input_video_path)
    width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
    fps_in = cap.get(cv.CAP_PROP_FPS)
    total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
    
    print(f"Memproses Video: {input_video_path}")
    print(f"Resolusi: {width}x{height} | FPS Asli: {fps_in:.1f} | Total Frame: {total_frames}")
    
    # Setup VideoWriter
    fourcc = cv.VideoWriter_fourcc(*'mp4v')
    out = cv.VideoWriter(output_video_path, fourcc, fps_in if fps_in > 0 else 30.0, (width, height))
    
    frame_count = 0
    start_time = time.time()
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        frame_start = time.time()
        
        # Inferensi YOLOv8 pada single frame
        results = model(frame, conf=conf_thresh, verbose=False)
        
        # Dapatkan frame yang sudah di-annotated oleh Ultralytics
        annotated_frame = results[0].plot()
        
        # Hitung FPS Real-time
        frame_time = time.time() - frame_start
        current_fps = 1.0 / frame_time if frame_time > 0 else 0
        
        # Overlay FPS pada Video
        fps_text = f"FPS: {current_fps:.1f}"
        cv.putText(annotated_frame, fps_text, (20, 50), cv.FONT_HERSHEY_SIMPLEX, 
                   1.0, (0, 255, 0), 2, cv.LINE_AA)
        
        # Tulis ke Video Output
        out.write(annotated_frame)
        frame_count += 1
        
        if frame_count % 30 == 0 or frame_count == total_frames:
            print(f"Progress: {frame_count}/{total_frames} frame ({(frame_count/total_frames)*100:.1f}%) | FPS: {current_fps:.1f}")
            
    cap.release()
    out.release()
    
    elapsed_time = time.time() - start_time
    print(f"\nPemrosesan video selesai dalam {elapsed_time:.2f} detik!")
    print(f"Video anotasi disimpan di: {output_video_path}")
    return output_video_path

# Contoh Penggunaan:
input_sample_video = '../week5/videoplayback.mp4'
output_sample_video = 'output_prediction.mp4'

if os.path.exists(input_sample_video):
    process_video_pipeline(input_sample_video, output_sample_video, model_path=best_model_path, conf_thresh=0.25)
else:
    print(f"Info: Silakan masukkan path file video Anda ke variabel `input_sample_video`.")

## Langkah 10: Antarmuka Interaktif Pemrosesan Video (Gradio Web UI)

Kita membangun antarmuka Web GUI menggunakan **Gradio**. Pengguna dapat mengunggah video apa saja, menggeser slider *Confidence Threshold*, dan melihat video hasil deteksi secara interaktif.

In [ ]:
import gradio as gr

def gradio_predict_video(video_file, conf_threshold):
    """
    Fungsi wrapper untuk Gradio Interface.
    """
    if video_file is None:
        return None
        
    output_path = "gradio_output_detected.mp4"
    process_video_pipeline(video_file, output_path, model_path=best_model_path, conf_thresh=conf_threshold)
    return output_path

# Membangun Tampilan UI Gradio
demo = gr.Interface(
    fn=gradio_predict_video,
    inputs=[
        gr.Video(label="Unggah Video Input (Underwater / Amarine Vision)"),
        gr.Slider(minimum=0.1, maximum=1.0, value=0.25, step=0.05, label="Confidence Threshold")
    ],
    outputs=gr.Video(label="Video Hasil Deteksi YOLOv8"),
    title="🚀 Amarine Vision - Custom YOLOv8 Video Detection Interface",
    description="Unggah berkas video Anda untuk menguji performa model kustom YOLOv8 (reels-moi4j v4) secara interaktif.",
    theme="soft"
)

# Jalankan Web Interface
demo.launch(inbrowser=True)